# Qwen2.5-1.5B Ablation Study — 完整实验流水线

**执行顺序（按优先级，Colab 随时断连也不怕）**：

| Phase | 内容 | 预计耗时 | 优先级 |
|-------|------|----------|--------|
| **P0** | 环境 + Drive + 数据同步 | ~5 min | 必须 |
| **P1a** | Group B SFT GSM8K 评测（产出 badcase） | ~20 min | 核心链路 |
| **P2** | 错误分类 + Targeted DPO 数据生成 | ~30 min | 核心链路 |
| **P4** | Group D 训练（Error-Type-Targeted DPO）+ 评测 | ~3.5h | 核心链路 |
| **P1b** | Group B 剩余评测（MATH+BBH）+ Group B DPO 全评测 | ~1.5h | 补充 |
| **P3** | Group A 训练（LoRA baseline）+ 评测 | ~4h | 补充 |
| **P5** | 汇总 + 同步 | ~2 min | 收尾 |

**核心链路**：P1a → P2 → P4（创新点验证优先完成）
**补充链路**：P1b + P3（基线对比，可后续补跑）

**断点续跑**：每个 Phase 检查 checkpoint，已完成跳过；评测每 20 条自动同步 Drive。
**前提**：Drive `Qwen-Reasoning/` 下有 `outputs/sft_merged/` + Colab Secrets 有 `HF_TOKEN`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P0: 环境准备 + Drive 挂载 + 代码克隆 + 数据同步
# 预计耗时：~5 min
# ═══════════════════════════════════════════════════════════════════

# ── 0.1 安装依赖 ──────────────────────────────────────────────────
!pip install -q -U pip
!pip install -q "unsloth>=2025.1.0" "trl>=0.14.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" \
    "transformers>=4.49.0" "datasets>=3.2.0" "accelerate>=1.2.0" \
    "pyyaml>=6.0.2" "safetensors" "tqdm" "scipy" "matplotlib" "seaborn"

import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB')

# ── 全局常量 & 工具函数 ──────────────────────────────────────────
BIT = ['--load_in_4bit']

def run_eval(cmd, label):
    """运行评测子进程，实时打印输出，后台每 30s 同步到 Drive。"""
    import subprocess, threading, time
    cmd = [cmd[0], '-u'] + cmd[1:]
    print(f'  执行: {" ".join(cmd[-6:])}')
    stop_event = threading.Event()
    def bg_sync():
        while not stop_event.is_set():
            time.sleep(30)
            try:
                subprocess.run(['rsync', '-av', '--include', '*.json', '--include', '*.jsonl',
                                '--include', '*/', '--exclude', '*',
                                'logs/', f'{DRIVE_DIR}/logs/'],
                               capture_output=True, timeout=60)
            except Exception:
                pass
    threading.Thread(target=bg_sync, daemon=True).start()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    stop_event.set()
    subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
    if proc.returncode != 0:
        print(f'  ❌ {label} 失败 (exit {proc.returncode})')
        return False
    return True

In [ ]:
# ── 0.2 挂载 Drive + 加载 Secrets ─────────────────────────────────
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/Qwen-Reasoning'

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN: OK')
except Exception:
    print('HF_TOKEN: missing（HuggingFace 下载可能受限）')

try:
    os.environ['DASHSCOPE_API_KEY'] = userdata.get('DASHSCOPE_API_KEY')
    print('DASHSCOPE_API_KEY: OK（P2 错误分类可用）')
except Exception:
    print('DASHSCOPE_API_KEY: missing（P2 将跳过，需本地运行 L3+L4）')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# ── 0.3 克隆代码 + 软链 outputs + 诊断 Drive ─────────────────────
import subprocess, shutil, glob as _glob

PROJECT_DIR = '/content/6000Q-QwenMiniReason'
REPO_URL = 'https://github.com/yukiiii0730/6000Q-QwenMiniReason.git'

if not os.path.isdir(f'{PROJECT_DIR}/.git'):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=False)

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, f'{PROJECT_DIR}/eval')
sys.path.insert(0, f'{PROJECT_DIR}/scripts')

# outputs/ 软链到 Drive（训练产物实时持久化）
out_dir = f'{PROJECT_DIR}/outputs'
if not os.path.islink(out_dir):
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.symlink(f'{DRIVE_DIR}/outputs', out_dir)
print(f'outputs/ -> {os.readlink(out_dir)}')

# 同步 Drive 的 logs 到项目目录
os.makedirs('logs', exist_ok=True)
subprocess.run(['rsync', '-av', f'{DRIVE_DIR}/logs/', 'logs/'], capture_output=True)

# 诊断 Drive 目录结构
print('\n📂 Drive 目录结构:')
for d in ['outputs', 'data/processed', 'logs']:
    full = f'{DRIVE_DIR}/{d}'
    if os.path.isdir(full):
        items = os.listdir(full)
        print(f'  {d}/ ({len(items)} items): {items[:10]}')
    else:
        print(f'  {d}/: NOT FOUND')
zips = _glob.glob(f'{DRIVE_DIR}/*.zip') + _glob.glob(f'{DRIVE_DIR}/**/*.zip', recursive=True)
if zips:
    print(f'\n📦 Drive 中的 zip 文件:')
    for z in zips[:10]:
        size = os.path.getsize(z) / 1024 / 1024
        print(f'  {os.path.basename(z):50s} {size:.1f} MB')

In [ ]:
# ── 0.4 同步数据 + 检查模型（自动搜索 + 解压 zip）──────────────────
import json, zipfile

os.chdir(PROJECT_DIR)

# ── 数据同步 ─────────────────────────────────────────────────────
os.makedirs('data/processed', exist_ok=True)
drive_data = f'{DRIVE_DIR}/data/processed'
if os.path.isdir(drive_data):
    subprocess.run(['rsync', '-av', f'{drive_data}/', 'data/processed/'],
                   capture_output=True)
    print(f'✅ 已从 Drive 同步 data/processed/')

# 如果 Drive 没有 data/processed，尝试解压 Drive 中的 zip
if not os.listdir('data/processed'):
    for zf in _glob.glob(f'{DRIVE_DIR}/*data*.zip') + _glob.glob(f'{DRIVE_DIR}/*processed*.zip'):
        print(f'📦 解压数据: {os.path.basename(zf)}')
        with zipfile.ZipFile(zf) as z:
            z.extractall(f'{DRIVE_DIR}/')
        if os.path.isdir(drive_data):
            subprocess.run(['rsync', '-av', f'{drive_data}/', 'data/processed/'],
                           capture_output=True)
        break

# ── 模型检查（先通过 symlink，再直接搜索 Drive）──────────────────
SFT_MERGED = None
DPO_MERGED = None
SFT_ADAPTER = None
DPO_ADAPTER = None
GROUPC_MERGED = None

candidates = {
    'SFT_MERGED': ['outputs/sft_merged', f'{DRIVE_DIR}/outputs/sft_merged'],
    'DPO_MERGED': ['outputs/merged',     f'{DRIVE_DIR}/outputs/merged'],
    'SFT_ADAPTER': ['outputs/sft',       f'{DRIVE_DIR}/outputs/sft'],
    'DPO_ADAPTER': ['outputs/dpo',       f'{DRIVE_DIR}/outputs/dpo'],
}
for var, paths in candidates.items():
    for p in paths:
        if os.path.isfile(f'{p}/config.json') or os.path.isfile(f'{p}/adapter_config.json'):
            globals()[var] = p
            break

for p in ['outputs/groupc_merged', f'{DRIVE_DIR}/outputs/groupc_merged']:
    if os.path.isfile(f'{p}/config.json'):
        GROUPC_MERGED = p
        break

print('\n📋 模型状态:')
for label, var in [('Group B SFT', SFT_MERGED), ('Group B DPO', DPO_MERGED),
                    ('SFT adapter', SFT_ADAPTER), ('DPO adapter', DPO_ADAPTER),
                    ('Group C', GROUPC_MERGED)]:
    status = 'OK' if var else 'MISSING'
    path = var or '—'
    print(f'  {label:20s}: {status} ({path})')

# NF4 量化检测：如果 merged 模型有 uint8 权重，需要 adapter 做 fallback
if SFT_MERGED:
    from model_loader import _has_quantized_weights, _find_adapter_dir
    if _has_quantized_weights(SFT_MERGED):
        adapters = _find_adapter_dir(SFT_MERGED)
        if adapters:
            print(f'\n⚠️ sft_merged 为 NF4 量化格式，将自动用 base + adapter 加载')
            print(f'   adapter: {adapters}')
        else:
            print(f'\n❌ sft_merged 为 NF4 量化格式且未找到 adapter！')
            print(f'   需要 {DRIVE_DIR}/outputs/sft/adapter_config.json')
            print(f'   请从训练环境同步 adapter 到 Drive')

# 如果 outputs 为空但 Drive 有 zip，自动解压
if not any([SFT_MERGED, DPO_MERGED]):
    drive_out = f'{DRIVE_DIR}/outputs'
    if not os.path.isdir(drive_out) or not os.listdir(drive_out):
        for zf in _glob.glob(f'{DRIVE_DIR}/*output*.zip') + _glob.glob(f'{DRIVE_DIR}/*merged*.zip'):
            print(f'\n📦 解压模型: {os.path.basename(zf)}')
            with zipfile.ZipFile(zf) as z:
                z.extractall(f'{DRIVE_DIR}/')
            break
        for var, paths in candidates.items():
            for p in paths:
                if os.path.isfile(f'{p}/config.json') or os.path.isfile(f'{p}/adapter_config.json'):
                    globals()[var] = p

# ── 训练数据检查 ─────────────────────────────────────────────────
print('\n📋 训练数据:')
for f in ['sft_train.json', 'dpo_train.json', 'dpo_teacher_round_1.json']:
    fp = f'data/processed/{f}'
    if os.path.isfile(fp):
        size = os.path.getsize(fp) / 1024 / 1024
        print(f'  {f:35s}: {size:.1f} MB')
    else:
        print(f'  {f:35s}: MISSING')

# ── A100 batch size ──────────────────────────────────────────────
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 4 if vram > 70 else 2
print(f'\nA100 batch_size={BATCH_SIZE}, grad_accum=8, effective_BS={BATCH_SIZE * 8}')

# 最终诊断
missing_critical = []
if not SFT_MERGED: missing_critical.append('outputs/sft_merged (Group B SFT)')
if not DPO_MERGED: missing_critical.append('outputs/merged (Group B DPO)')
missing_data = [f for f in ['sft_train.json', 'dpo_train.json']
                if not os.path.isfile(f'data/processed/{f}')]
if missing_critical or missing_data:
    print('\n⚠️ 缺少关键文件:')
    for m in missing_critical + missing_data:
        print(f'  ❌ {m}')
    print(f'\n请确保 Google Drive 中有:')
    print(f'  {DRIVE_DIR}/outputs/sft_merged/config.json')
    print(f'  {DRIVE_DIR}/outputs/merged/config.json')
    print(f'  {DRIVE_DIR}/data/processed/sft_train.json')
    print(f'  {DRIVE_DIR}/data/processed/dpo_train.json')
else:
    print('\n✅ 所有关键文件就位，可以开始 P1')

---
## P1a: Group B SFT GSM8K 评测（核心链路第一步）

只跑 GSM8K n=200，产出 badcase 文件供 P2 错误分类用。
MATH + BBH 留到 P1b 再跑。

预计耗时：~20 min

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P1a: Group B SFT GSM8K 评测（产出 badcase 给 P2 用）
# 断点续跑 + 实时输出 + 自动同步 Drive
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, json, shutil

os.chdir(PROJECT_DIR)
GSM8K_N = '200'

# 从 Drive 恢复 checkpoint
drive_logs = f'{DRIVE_DIR}/logs'
if os.path.isdir(drive_logs):
    restored = 0
    for f in os.listdir(drive_logs):
        if f.endswith('.json') or f.endswith('.jsonl'):
            src, dst = os.path.join(drive_logs, f), f'logs/{f}'
            if os.path.isfile(src) and not os.path.isfile(dst):
                shutil.copy2(src, dst)
                restored += 1
    if restored:
        print(f'  📂 从 Drive 恢复 {restored} 个文件')


# ── Group B SFT GSM8K ────────────────────────────────────────
if not SFT_MERGED:
    print('⚠️ Group B SFT 模型不存在，跳过 P1a')
else:
    gsm_out = 'logs/gsm8k_sft.json'
    gsm_bc = 'logs/gsm8k_sft_badcases.jsonl'
    done = False
    if os.path.isfile(gsm_out):
        with open(gsm_out) as f:
            r = json.load(f)
        total = r.get('total', 0)
        if total >= int(GSM8K_N):
            print(f'GSM8K SFT: already done ({r.get("accuracy", 0):.1%}, {total} 条)')
            done = True
        else:
            print(f'GSM8K SFT: checkpoint ({total}/{GSM8K_N})，续跑...')
    if not done:
        run_eval([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', SFT_MERGED,
            '--max_samples', GSM8K_N,
            '--sampling_mode', 'stratified',
            '--output', gsm_out,
            '--badcase_output', gsm_bc,
        ] + BIT, 'GSM8K SFT')
    if os.path.isfile(gsm_out):
        d = json.load(open(gsm_out))
        print(f'  GSM8K SFT 结果: {d.get("accuracy", 0):.1%} ({d.get("correct", 0)}/{d.get("total", 0)})')
    if os.path.isfile(gsm_bc):
        n = sum(1 for _ in open(gsm_bc))
        print(f'  Badcase: {n} 条 → {gsm_bc}')

print('\nP1a 完成')

---
## P2: 错误分类 + Targeted DPO 数据生成（核心链路第二步）

用 qwen-flash 对 SFT badcase 做 5 类错误分类，再用 qwen3-235b 生成类型专属 DPO 数据。

**前提**：P1a 已产出 `logs/gsm8k_sft_badcases.jsonl`
**需要**：`DASHSCOPE_API_KEY`（无则跳过，需本地运行 L3+L4）

预计耗时：~30 min

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P2: 错误分类 + Targeted DPO 数据生成
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, json

os.chdir(PROJECT_DIR)

# 检查是否已有 targeted DPO 数据（从 Drive 同步的或之前生成的）
TARGETED_DPO = 'data/processed/dpo_targeted_round1.json'
if os.path.isfile(TARGETED_DPO):
    data = json.load(open(TARGETED_DPO))
    print(f'dpo_targeted_round1.json 已存在 ({len(data)} 条)，跳过 P2')
else:
    api_key = os.environ.get('DASHSCOPE_API_KEY', '')
    if not api_key:
        print('DASHSCOPE_API_KEY 未设置，跳过 P2')
        print('请在本地运行: bash run_local_pipeline.sh')
        print('然后上传 data/processed/dpo_targeted_round1.json 到 Drive')
    else:
        # ── L3: 错误分类 ───────────────────────────────────────────
        BADCASE = 'logs/gsm8k_sft_badcases.jsonl'
        if not os.path.isfile(BADCASE):
            print(f'找不到 {BADCASE}，请先完成 P1 的 SFT 评测')
        else:
            print(f'L3: 错误分类 ({BADCASE}) ...')
            subprocess.run([
                'python3', 'scripts/classify_errors.py',
                '--badcase_jsonl', BADCASE,
                '--output_dir', 'results/errors/sft',
                '--workers', '4',
            ], check=True)
            print('L3 完成')

            # ── L4: 生成 Targeted DPO 数据 ─────────────────────────
            print(f'L4: 生成 Targeted DPO 数据 ...')
            subprocess.run([
                'python3', 'scripts/build_targeted_dpo.py',
                '--by_type_dir', 'results/errors/sft/by_type',
                '--per_type_n', '200',
                '--tag', 'round1',
                '--output_dir', 'data/processed',
                '--workers', '4',
            ], check=True)

            if os.path.isfile(TARGETED_DPO):
                data = json.load(open(TARGETED_DPO))
                print(f'L4 完成: {len(data)} 条 targeted DPO 数据')
                # 同步到 Drive
                subprocess.run(['rsync', '-av', 'data/processed/',
                                f'{DRIVE_DIR}/data/processed/'], capture_output=True)
                subprocess.run(['rsync', '-av', 'results/',
                                f'{DRIVE_DIR}/results/'], capture_output=True)
                print('数据已同步到 Drive')

---
## P4: Group D 训练（Error-Type-Targeted DPO）（核心链路第三步）

**创新核心**：用 P2 生成的 targeted DPO 数据 + error_type 加权训练。

**前提**：`data/processed/dpo_targeted_round1.json` 已存在（P2 生成或本地上传）。

| 配置 | Group D | Group B |
|------|---------|--------|
| SFT | DoRA + 五段课程（共用）| DoRA + 五段课程 |
| DPO 数据 | **Targeted（类型专属）** | distilabel 通用 |
| DPO loss | sigmoid + **error_type 加权** | sigmoid |

预计耗时：~3.5h（训练 ~2h + 评测 ~1.5h）

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P4: Group D 训练（Error-Type-Targeted DPO）
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, yaml, json

os.chdir(PROJECT_DIR)

G_D_DPO = 'outputs/group_d/dpo'
G_D_MERGED = 'outputs/group_d/merged'
TARGETED_DPO = 'data/processed/dpo_targeted_round1.json'

if not SFT_MERGED:
    print('SFT 基座不存在，请先完成 P1a 或检查 Drive 中的 sft_merged')
elif not os.path.isfile(TARGETED_DPO):
    print(f'{TARGETED_DPO} 不存在')
    print('请先完成 P2，或在本地运行 bash run_local_pipeline.sh 后上传到 Drive')
elif os.path.isfile(f'{G_D_MERGED}/config.json'):
    print('Group D merged 已存在，跳过训练')
else:
    data = json.load(open(TARGETED_DPO))
    print(f'Targeted DPO 数据: {len(data)} 条')
    if 'error_type' in data[0]:
        from collections import Counter
        types = Counter(d.get('error_type', '?') for d in data)
        for t, n in types.most_common():
            print(f'  {t}: {n}')

    print('\nGroup D DPO 训练（Error-Type-Targeted，~2h on A100）...')
    dpo_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'base_adapter_path': SFT_MERGED,
        'output_dir': G_D_DPO,
        'max_seq_length': 2048,
        'load_in_4bit': True,
        'seed': 42,
        'beta': 0.1,
        'loss_type': 'sigmoid',
        'error_type_weights': {
            'arithmetic': 1, 'reasoning_skip': 2, 'setup_error': 3,
            'unit_or_format': 1, 'extraction_error': 1,
        },
        'dataset': {'name': 'local', 'split': 'train', 'max_samples': -1},
        'lora': {
            'use_dora': True, 'r': 16, 'alpha': 32, 'dropout': 0.0,
            'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        },
        'train': {
            'per_device_train_batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': 8,
            'warmup_steps': 50, 'max_steps': 600, 'learning_rate': 1e-5,
            'logging_steps': 10, 'save_steps': 100, 'weight_decay': 0.0,
            'lr_scheduler_type': 'cosine', 'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True, 'dataloader_num_workers': 4,
        },
        'dataset_path': TARGETED_DPO,
    }
    os.makedirs('config', exist_ok=True)
    with open('config/dpo_config_group_d.yaml', 'w') as f:
        yaml.dump(dpo_cfg, f, allow_unicode=True)

    subprocess.run(['python3', 'scripts/dpo_train.py', '--config', 'config/dpo_config_group_d.yaml'], check=True)

    print('合并 Group D adapter ...')
    os.makedirs(G_D_MERGED, exist_ok=True)
    subprocess.run(['python3', 'scripts/merge_lora.py', '--adapter_path', G_D_DPO, '--output_path', G_D_MERGED], check=True)

    subprocess.run(['rsync', '-av', 'outputs/group_d/', f'{DRIVE_DIR}/outputs/group_d/'], capture_output=True)
    print('Group D 训练完成')

In [ ]:
# ── P4.2: Group D 评测（断点续跑 + 实时输出 + 自动同步 Drive）──
import subprocess, os, json, glob, shutil

os.chdir(PROJECT_DIR)

if not os.path.isfile(f'{G_D_MERGED}/config.json'):
    print('Group D 模型不存在，跳过评测')
else:
    print('Group D 评测 ...')
    for tag, script, n in [('gsm8k', 'eval/gsm8k_eval.py', '200'), ('math', 'eval/math_eval.py', '200')]:
        out = f'logs/group_d_{tag}.json'
        if os.path.isfile(out):
            r = json.load(open(out))
            if r.get('total', 0) >= int(n):
                print(f'  {tag}: already done ({r.get("accuracy", 0):.1%})')
                continue
        cmd = ['python3', script, '--model_path', G_D_MERGED, '--max_samples', n, '--output', out]
        if tag == 'gsm8k':
            cmd += ['--sampling_mode', 'stratified', '--badcase_output', 'logs/group_d_gsm8k_badcases.jsonl']
        run_eval(cmd + BIT, f'Group D {tag}')

    bbh_out = 'logs/group_d_bbh.json'
    if not os.path.isfile(bbh_out):
        run_eval(['python3', 'eval/bbh_full_eval.py', '--mode', 'local',
                  '--model_path', G_D_MERGED, '--max_samples', '30',
                  '--output_dir', 'logs/group_d_bbh_eval'] + BIT, 'Group D BBH')
        sums = glob.glob('logs/group_d_bbh_eval/*summary*')
        if sums: shutil.copy2(sums[0], bbh_out)

    for name, path in [('GSM8K', 'logs/group_d_gsm8k.json'), ('MATH', 'logs/group_d_math.json'), ('BBH', bbh_out)]:
        if os.path.isfile(path):
            d = json.load(open(path))
            acc = d.get('accuracy') or d.get('macro_avg_accuracy', 0)
            print(f'  {name}: {acc:.2%}')

print('\nP4 完成')

---
## P1b: Group B 剩余评测（MATH + BBH + DPO 全量）

补充完成 Group B 的 MATH-500、BBH-27 评测（SFT 和 DPO 两个模型）。

预计耗时：~1.5h

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P1b: Group B 剩余评测（MATH + BBH + DPO 全量）
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, json, glob, shutil

os.chdir(PROJECT_DIR)


def eval_remaining(label, model_path, tag):
    """跑 MATH + BBH（GSM8K 已在 P1a 完成）。"""
    print(f'\n{"="*60}\n  评测 {label} 剩余 (MATH+BBH)\n{"="*60}')

    # MATH-500
    math_out = f'logs/math_{tag}.json'
    if os.path.isfile(math_out):
        r = json.load(open(math_out))
        if r.get('total', 0) >= 200:
            print(f'  MATH: already done ({r.get("accuracy", 0):.1%})')
        else:
            run_eval(['python3', 'eval/math_eval.py', '--model_path', model_path,
                       '--max_samples', '200', '--output', math_out] + BIT, f'{label} MATH')
    else:
        run_eval(['python3', 'eval/math_eval.py', '--model_path', model_path,
                   '--max_samples', '200', '--output', math_out] + BIT, f'{label} MATH')

    # BBH-27
    bbh_out = f'logs/bbh_{tag}.json'
    bbh_dir = f'logs/bbh_{tag}_eval'
    if os.path.isfile(bbh_out):
        print(f'  BBH: already done')
    else:
        run_eval(['python3', 'eval/bbh_full_eval.py', '--mode', 'local',
                   '--model_path', model_path, '--max_samples', '30',
                   '--output_dir', bbh_dir] + BIT, f'{label} BBH')
        sums = glob.glob(f'{bbh_dir}/*summary*')
        if sums: shutil.copy2(sums[0], bbh_out)

    for name, path in [('MATH', math_out), ('BBH', bbh_out)]:
        if os.path.isfile(path):
            d = json.load(open(path))
            acc = d.get('accuracy') or d.get('macro_avg_accuracy', 0)
            print(f'  {name}: {acc:.2%}')


# ── Group B SFT 剩余（MATH + BBH）────────────────────────────
if SFT_MERGED:
    eval_remaining('Group B SFT', SFT_MERGED, 'sft')
else:
    print('⚠️ SFT 模型不存在，跳过')

# ── Group B DPO 全量 ──────────────────────────────────────────
if DPO_MERGED:
    print(f'\n{"="*60}\n  评测 Group B SFT+DPO (全量)\n{"="*60}')
    for tag, script, n, bc in [
        ('gsm8k', 'eval/gsm8k_eval.py', '200', True),
        ('math', 'eval/math_eval.py', '200', False),
    ]:
        out = f'logs/{tag}_result.json'
        if os.path.isfile(out):
            r = json.load(open(out))
            if r.get('total', 0) >= int(n):
                print(f'  {tag}: already done ({r.get("accuracy", 0):.1%})')
                continue
        cmd = ['python3', script, '--model_path', DPO_MERGED, '--max_samples', n, '--output', out]
        if tag == 'gsm8k':
            cmd += ['--sampling_mode', 'stratified']
        run_eval(cmd + BIT, f'DPO {tag}')

    bbh_out = 'logs/bbh_result.json'
    if not os.path.isfile(bbh_out):
        run_eval(['python3', 'eval/bbh_full_eval.py', '--mode', 'local',
                   '--model_path', DPO_MERGED, '--max_samples', '30',
                   '--output_dir', 'logs/bbh_result_eval'] + BIT, 'DPO BBH')
        sums = glob.glob('logs/bbh_result_eval/*summary*')
        if sums: shutil.copy2(sums[0], bbh_out)

    for name, path in [('GSM8K', 'logs/gsm8k_result.json'), ('MATH', 'logs/math_result.json'), ('BBH', bbh_out)]:
        if os.path.isfile(path):
            d = json.load(open(path))
            acc = d.get('accuracy') or d.get('macro_avg_accuracy', 0)
            print(f'  {name}: {acc:.2%}')
else:
    print('⚠️ DPO 模型不存在，跳过')

subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
print('\nP1b 完成')

---
## P3: Group A 训练（LoRA + 单段混合 SFT + Standard DPO）

经典 baseline，与 Group B（DoRA + 五段课程）对比。

| 配置 | Group A | Group B |
|------|---------|--------|
| PEFT | LoRA | DoRA |
| SFT 数据 | 单段混合 15k | 五段课程 38k |
| DPO 数据 | distilabel 5k | distilabel 5k |
| DPO loss | sigmoid | sigmoid |

预计耗时：~4h（SFT ~2.5h + DPO ~1.5h + 评测 ~30min）

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P3: Group A 训练 + 评测
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, yaml, json, glob, shutil

os.chdir(PROJECT_DIR)

G_A_SFT = 'outputs/group_a/sft'
G_A_SFT_MERGED = 'outputs/group_a/sft_merged'
G_A_DPO = 'outputs/group_a/dpo'
G_A_MERGED = 'outputs/group_a/merged'


# ── P3.1: Group A SFT ────────────────────────────────────────
if os.path.isfile(f'{G_A_SFT_MERGED}/config.json'):
    print('Group A sft_merged 已存在，跳过 SFT 训练')
else:
    print('Group A SFT 训练（LoRA + 单段混合，~2.5h on A100）...')
    sft_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'output_dir': G_A_SFT,
        'max_seq_length': 2048, 'load_in_4bit': True, 'seed': 42,
        'stages': [{'name': 'stage_mixed', 'dataset': {
            'path': 'data/processed/sft_train.json', 'max_samples': 15000,
            'field_map': {'prompt': 'input', 'response': 'output'},
        }, 'train': {'max_steps': 900, 'learning_rate': 3e-5, 'warmup_steps': 90}}],
        'lora': {'use_dora': False, 'r': 16, 'alpha': 32, 'dropout': 0.0,
                 'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']},
        'train': {
            'per_device_train_batch_size': BATCH_SIZE, 'gradient_accumulation_steps': 8,
            'warmup_steps': 90, 'max_steps': 900, 'learning_rate': 3e-5,
            'logging_steps': 10, 'save_steps': 300, 'weight_decay': 0.01,
            'lr_scheduler_type': 'cosine', 'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True, 'dataloader_num_workers': 4, 'packing': True,
        },
        'dataset_path': 'data/processed/sft_train.json',
    }
    os.makedirs('config', exist_ok=True)
    with open('config/sft_config_group_a.yaml', 'w') as f:
        yaml.dump(sft_cfg, f, allow_unicode=True)
    subprocess.run(['python3', 'scripts/sft_train.py', '--config', 'config/sft_config_group_a.yaml'], check=True)

    print('合并 Group A SFT adapter ...')
    os.makedirs(G_A_SFT_MERGED, exist_ok=True)
    subprocess.run(['python3', 'scripts/merge_lora.py', '--adapter_path', G_A_SFT, '--output_path', G_A_SFT_MERGED], check=True)

subprocess.run(['rsync', '-av', 'outputs/group_a/', f'{DRIVE_DIR}/outputs/group_a/'], capture_output=True)
print('Group A SFT 完成')


# ── P3.2: Group A DPO ────────────────────────────────────────
if os.path.isfile(f'{G_A_MERGED}/config.json'):
    print('Group A merged 已存在，跳过 DPO')
else:
    print('Group A DPO 训练（Standard sigmoid，~1.5h on A100）...')
    dpo_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'base_adapter_path': G_A_SFT_MERGED,
        'output_dir': G_A_DPO,
        'max_seq_length': 2048, 'load_in_4bit': True, 'seed': 42, 'beta': 0.1, 'loss_type': 'sigmoid',
        'dataset': {'name': 'argilla/distilabel-math-preference-dpo', 'split': 'train', 'max_samples': 5000},
        'lora': {'use_dora': False, 'r': 16, 'alpha': 32, 'dropout': 0.0,
                 'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']},
        'train': {
            'per_device_train_batch_size': BATCH_SIZE, 'gradient_accumulation_steps': 8,
            'warmup_steps': 50, 'max_steps': 600, 'learning_rate': 1e-5,
            'logging_steps': 10, 'save_steps': 100, 'weight_decay': 0.0,
            'lr_scheduler_type': 'cosine', 'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True, 'dataloader_num_workers': 4,
        },
        'dataset_path': 'data/processed/dpo_train.json',
    }
    with open('config/dpo_config_group_a.yaml', 'w') as f:
        yaml.dump(dpo_cfg, f, allow_unicode=True)
    subprocess.run(['python3', 'scripts/dpo_train.py', '--config', 'config/dpo_config_group_a.yaml'], check=True)

    print('合并 Group A DPO adapter ...')
    os.makedirs(G_A_MERGED, exist_ok=True)
    subprocess.run(['python3', 'scripts/merge_lora.py', '--adapter_path', G_A_DPO, '--output_path', G_A_MERGED], check=True)

subprocess.run(['rsync', '-av', 'outputs/group_a/', f'{DRIVE_DIR}/outputs/group_a/'], capture_output=True)
print('Group A DPO 完成')


# ── P3.3: Group A 评测（断点续跑 + 实时输出 + 自动同步 Drive）──
def eval_group_a(label, model_path, tag):
    print(f'\n{"="*60}\n  评测 {label}\n{"="*60}')
    for ds, script, n in [('gsm8k', 'eval/gsm8k_eval.py', '200'), ('math', 'eval/math_eval.py', '200')]:
        out = f'logs/group_a_{tag}_{ds}.json'
        if os.path.isfile(out):
            r = json.load(open(out))
            if r.get('total', 0) >= int(n):
                print(f'  {ds}: already done ({r.get("accuracy", 0):.1%})')
                continue
        cmd = ['python3', script, '--model_path', model_path, '--max_samples', n, '--output', out]
        if ds == 'gsm8k':
            cmd += ['--sampling_mode', 'stratified']
            if tag == 'sft':
                cmd += ['--badcase_output', f'logs/group_a_sft_gsm8k_badcases.jsonl']
        run_eval(cmd + BIT, f'{label} {ds}')

    bbh_out = f'logs/group_a_{tag}_bbh.json'
    if not os.path.isfile(bbh_out):
        run_eval(['python3', 'eval/bbh_full_eval.py', '--mode', 'local',
                   '--model_path', model_path, '--max_samples', '30',
                   '--output_dir', f'logs/group_a_{tag}_bbh_eval'] + BIT, f'{label} BBH')
        sums = glob.glob(f'logs/group_a_{tag}_bbh_eval/*summary*')
        if sums: shutil.copy2(sums[0], bbh_out)

    for name, path in [('GSM8K', f'logs/group_a_{tag}_gsm8k.json'), ('MATH', f'logs/group_a_{tag}_math.json'), ('BBH', bbh_out)]:
        if os.path.isfile(path):
            d = json.load(open(path))
            acc = d.get('accuracy') or d.get('macro_avg_accuracy', 0)
            print(f'  {name}: {acc:.2%}')


if os.path.isfile(f'{G_A_SFT_MERGED}/config.json'):
    eval_group_a('Group A SFT', G_A_SFT_MERGED, 'sft')
if os.path.isfile(f'{G_A_MERGED}/config.json'):
    eval_group_a('Group A DPO', G_A_MERGED, 'dpo')

subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
print('\nP3 完成')

---
## P5: 汇总 + 同步

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P5: 汇总所有实验结果
# ═══════════════════════════════════════════════════════════════════
import json, os, subprocess
from pathlib import Path

os.chdir(PROJECT_DIR)

def load_acc(path):
    if not Path(path).exists():
        return None
    d = json.loads(Path(path).read_text())
    return d.get('accuracy') or d.get('macro_avg_accuracy')

groups = [
    ('A', 'LoRA + 单段SFT + Std DPO',
     'logs/group_a_dpo_gsm8k.json', 'logs/group_a_dpo_math.json', 'logs/group_a_dpo_bbh.json'),
    ('B-SFT', 'DoRA + 五段课程 (SFT only)',
     'logs/gsm8k_sft.json', 'logs/math_sft.json', 'logs/bbh_sft.json'),
    ('B', 'DoRA + 五段课程 + Std DPO',
     'logs/gsm8k_result.json', 'logs/math_result.json', 'logs/bbh_result.json'),
    ('D', 'DoRA + 五段课程 + Targeted DPO',
     'logs/group_d_gsm8k.json', 'logs/group_d_math.json', 'logs/group_d_bbh.json'),
]

print('=' * 70)
print(f'{"Group":8} {"Description":40} {"GSM8K":>8} {"MATH":>8} {"BBH":>8}')
print('-' * 70)

rows = []
for g, desc, gsm, math, bbh in groups:
    ga, ma, ba = load_acc(gsm), load_acc(math), load_acc(bbh)
    g_str = f'{ga*100:.1f}%' if ga else '  N/A'
    m_str = f'{ma*100:.1f}%' if ma else '  N/A'
    b_str = f'{ba*100:.1f}%' if ba else '  N/A'
    print(f'{g:8} {desc:40} {g_str:>8} {m_str:>8} {b_str:>8}')
    rows.append({'group': g, 'desc': desc, 'gsm8k': ga, 'math': ma, 'bbh': ba})

print('=' * 70)

os.makedirs('results/ablation', exist_ok=True)
with open('results/ablation/summary_table.json', 'w') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

subprocess.run(['python3', 'eval/compare_table.py'], check=False)
subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
subprocess.run(['rsync', '-av', 'results/', f'{DRIVE_DIR}/results/'], capture_output=True)

print('\n全部完成！结果已同步到 Drive。')